# PIDA-RL Diabetes — 10. Equidad por nivel de riesgo

Auditoría exploratoria de las seis políticas con archivos de evaluación agregados por paso. Los datos disponibles no contienen sexo, edad, grupo etario ni identificador individual; por rigor, este notebook evalúa la distribución operacional por nivel de riesgo.

> Los resultados pertenecen al simulador. No prueban efectividad clínica, causalidad ni aptitud para despliegue institucional.


## Objetivo

Comparar cobertura, contactos, eventos adversos, costos y excesos de capacidad entre riesgo bajo, medio y alto. También se calcula si cada política prioriza alto riesgo respecto de bajo riesgo.


## Inicialización


In [ ]:
from pathlib import Path
import os
import subprocess
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

DRIVE_ROOT = next(
    (path for path in [Path('/content/drive'), Path('/content/gdrive')] if (path / 'MyDrive').is_dir()),
    None,
)

if DRIVE_ROOT is None:
    from google.colab import drive
    mount_point = Path('/content/drive')
    if mount_point.exists() and any(mount_point.iterdir()):
        mount_point = Path('/content/gdrive')
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError('No hay un punto de montaje vacío para Google Drive. Reinicia la sesión.')
    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / 'MyDrive'
PERSISTENT_ROOT = MYDRIVE_ROOT / 'PIDA-RL-Diabetes'
TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
FIGURES_DIR = PERSISTENT_ROOT / 'results' / 'figures'
METADATA_DIR = PERSISTENT_ROOT / 'data' / 'metadata'

for directory in [TABLES_DIR, FIGURES_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'La ruta no es un repositorio Git: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
print('Proyecto temporal:', PROJECT_ROOT)
print('Tablas:', TABLES_DIR)
print('Figuras:', FIGURES_DIR)


## Archivos de entrada


In [ ]:
STEP_FILES = {
    'politicas_base': TABLES_DIR / 'evaluacion_pasos_politicas_base_v1.csv',
    'dqn_v1': TABLES_DIR / 'evaluacion_pasos_dqn_v1.csv',
    'q_learning_v1': TABLES_DIR / 'evaluacion_pasos_q_learning_v1.csv',
    'ppo_v1': TABLES_DIR / 'evaluacion_pasos_ppo_v1.csv',
}

for source_name, path in STEP_FILES.items():
    print(f'{source_name:20s} | existe: {path.is_file()} | {path}')

missing_files = [str(path) for path in STEP_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan archivos de evaluación por paso:\n' + '\n'.join(missing_files)
    )


## Carga y validación


In [ ]:
REQUIRED_COLUMNS = [
    'politica',
    'semilla',
    'paso',
    'mes',
    'nivel_riesgo',
    'categoria_riesgo',
    'accion',
    'accion_aceptada',
    'contacto_aceptado',
    'evento_adverso',
    'recompensa',
    'costo_real',
    'excede_capacidad',
]

frames = []
for source_name, path in STEP_FILES.items():
    frame = pd.read_csv(path)
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
    if missing_columns:
        raise KeyError(
            f'{path.name}: faltan columnas requeridas: {missing_columns}. '
            f'Columnas disponibles: {list(frame.columns)}'
        )

    frame = frame[REQUIRED_COLUMNS].copy()
    frame['archivo_origen'] = path.name
    frame['fuente_evaluacion'] = source_name
    frames.append(frame)

df_pasos = pd.concat(frames, ignore_index=True)

df_pasos['politica'] = df_pasos['politica'].astype(str).str.strip()
df_pasos['categoria_riesgo'] = (
    df_pasos['categoria_riesgo']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'low': 'bajo', 'medium': 'medio', 'high': 'alto'})
)

numeric_columns = [
    'semilla',
    'paso',
    'mes',
    'nivel_riesgo',
    'accion',
    'accion_aceptada',
    'contacto_aceptado',
    'evento_adverso',
    'recompensa',
    'costo_real',
    'excede_capacidad',
]

for column in numeric_columns:
    df_pasos[column] = pd.to_numeric(df_pasos[column], errors='coerce')

if df_pasos[numeric_columns].isna().any().any():
    missing_detail = df_pasos[numeric_columns].isna().sum()
    missing_detail = missing_detail[missing_detail > 0].to_dict()
    raise ValueError(f'Hay valores no numéricos o faltantes en columnas críticas: {missing_detail}')

expected_policies = {
    'aleatoria',
    'reglas_riesgo',
    'reglas_capacidad',
    'dqn_v1',
    'q_learning_v1',
    'ppo_v1',
}
observed_policies = set(df_pasos['politica'].unique())
missing_policies = expected_policies - observed_policies
if missing_policies:
    raise ValueError(
        'Faltan políticas esperadas: ' + ', '.join(sorted(missing_policies))
        + '. Políticas observadas: ' + ', '.join(sorted(observed_policies))
    )

valid_risk_categories = {'bajo', 'medio', 'alto'}
if not df_pasos['categoria_riesgo'].isin(valid_risk_categories).all():
    unexpected = sorted(
        set(df_pasos.loc[~df_pasos['categoria_riesgo'].isin(valid_risk_categories), 'categoria_riesgo'])
    )
    raise ValueError(f'Categorías de riesgo no reconocidas: {unexpected}')

print('Filas consolidadas:', len(df_pasos))
print('Políticas:', sorted(observed_policies))
display(df_pasos.head())


## Validación de protocolo


In [ ]:
df_validacion_pasos = (
    df_pasos
    .groupby('politica', as_index=False)
    .agg(
        filas=('politica', 'size'),
        semillas=('semilla', 'nunique'),
        meses=('mes', 'nunique'),
        niveles_riesgo=('nivel_riesgo', 'nunique'),
        categorias_riesgo=('categoria_riesgo', 'nunique'),
        pasos_minimos=('paso', 'min'),
        pasos_maximos=('paso', 'max'),
    )
)

display(df_validacion_pasos)
assert (df_validacion_pasos['semillas'] == 3).all(), 'No todas las políticas tienen tres semillas.'
assert (df_validacion_pasos['niveles_riesgo'] == 3).all(), 'No todas las políticas tienen tres niveles de riesgo.'
assert (df_validacion_pasos['categorias_riesgo'] == 3).all(), 'No todas las políticas tienen las tres categorías de riesgo.'


## Indicadores por riesgo

Cada fila de entrada representa una decisión agregada por paso y nivel de riesgo. La cobertura es el porcentaje de filas donde `contacto_aceptado = 1` para cada política, semilla y nivel de riesgo.


In [ ]:
df_riesgo_semilla = (
    df_pasos
    .groupby(['politica', 'semilla', 'nivel_riesgo', 'categoria_riesgo'], as_index=False)
    .agg(
        decisiones=('contacto_aceptado', 'size'),
        contactos_aceptados=('contacto_aceptado', 'sum'),
        acciones_aceptadas=('accion_aceptada', 'sum'),
        eventos_adversos=('evento_adverso', 'sum'),
        recompensa_acumulada=('recompensa', 'sum'),
        costo_total=('costo_real', 'sum'),
        excesos_capacidad=('excede_capacidad', 'sum'),
    )
)

df_riesgo_semilla['cobertura_porcentaje'] = (
    100.0 * df_riesgo_semilla['contactos_aceptados'] / df_riesgo_semilla['decisiones']
)
df_riesgo_semilla['tasa_eventos_porcentaje'] = (
    100.0 * df_riesgo_semilla['eventos_adversos'] / df_riesgo_semilla['decisiones']
)

df_equidad_riesgo = (
    df_riesgo_semilla
    .groupby(['politica', 'nivel_riesgo', 'categoria_riesgo'], as_index=False)
    .agg(
        decisiones_medias=('decisiones', 'mean'),
        contactos_medios=('contactos_aceptados', 'mean'),
        cobertura_media=('cobertura_porcentaje', 'mean'),
        cobertura_desviacion=('cobertura_porcentaje', 'std'),
        tasa_eventos_media=('tasa_eventos_porcentaje', 'mean'),
        recompensa_media=('recompensa_acumulada', 'mean'),
        costo_medio=('costo_total', 'mean'),
        excesos_capacidad_medios=('excesos_capacidad', 'mean'),
    )
)

df_equidad_riesgo = df_equidad_riesgo.round(3)
display(df_equidad_riesgo.sort_values(['politica', 'nivel_riesgo']))


## Brechas entre riesgos

La brecha total de riesgo es la diferencia entre la cobertura máxima y mínima de los tres niveles. La brecha alto–bajo indica la diferencia de cobertura entre las categorías de mayor y menor riesgo.


In [ ]:
riesgo_pivot = (
    df_equidad_riesgo
    .pivot(index='politica', columns='categoria_riesgo', values='cobertura_media')
    .reindex(columns=['bajo', 'medio', 'alto'])
    .reset_index()
)

if riesgo_pivot[['bajo', 'medio', 'alto']].isna().any().any():
    raise ValueError('No hay cobertura válida para bajo, medio y alto riesgo en todas las políticas.')

riesgo_pivot['cobertura_riesgo_minima'] = riesgo_pivot[['bajo', 'medio', 'alto']].min(axis=1)
riesgo_pivot['cobertura_riesgo_maxima'] = riesgo_pivot[['bajo', 'medio', 'alto']].max(axis=1)
riesgo_pivot['brecha_absoluta_riesgo_pp'] = (
    riesgo_pivot['cobertura_riesgo_maxima'] - riesgo_pivot['cobertura_riesgo_minima']
)
riesgo_pivot['brecha_alto_menos_bajo_pp'] = riesgo_pivot['alto'] - riesgo_pivot['bajo']

df_brechas_riesgo = riesgo_pivot.sort_values('brecha_absoluta_riesgo_pp').round(3)
display(df_brechas_riesgo)


## Alto riesgo por semilla


In [ ]:
df_alto_riesgo_semilla = (
    df_riesgo_semilla
    .query('categoria_riesgo == "alto"')
    .sort_values(['politica', 'semilla'])
    .reset_index(drop=True)
)

if df_alto_riesgo_semilla.empty:
    raise ValueError('No hay registros de alto riesgo en los archivos de evaluación por paso.')

display(df_alto_riesgo_semilla[[
    'politica',
    'semilla',
    'decisiones',
    'contactos_aceptados',
    'cobertura_porcentaje',
    'eventos_adversos',
    'costo_total',
    'excesos_capacidad',
]].round(3))


## Integración con notebook 09


In [ ]:
RANKING_PATH = TABLES_DIR / 'ranking_tecnico_modelos_v1.csv'
if not RANKING_PATH.is_file():
    raise FileNotFoundError(f'No existe el ranking técnico del notebook 09: {RANKING_PATH}')

df_ranking_tecnico = pd.read_csv(RANKING_PATH)

df_resumen_equidad = (
    df_ranking_tecnico
    .merge(
        df_brechas_riesgo[[
            'politica',
            'bajo',
            'medio',
            'alto',
            'brecha_absoluta_riesgo_pp',
            'brecha_alto_menos_bajo_pp',
        ]],
        on='politica',
        how='left',
    )
)

df_resumen_equidad['alerta_cobertura_minima_riesgo'] = (
    df_resumen_equidad[['bajo', 'medio', 'alto']].min(axis=1) < 10.0
)
df_resumen_equidad['alerta_exceso_capacidad'] = (
    ~df_resumen_equidad['respeta_capacidad_en_promedio'].astype(bool)
)

display(df_resumen_equidad[[
    'ranking_tecnico',
    'politica',
    'clasificacion',
    'puntaje_tecnico',
    'cobertura_alto_riesgo_media',
    'respeta_capacidad_en_promedio',
    'bajo',
    'medio',
    'alto',
    'brecha_absoluta_riesgo_pp',
    'brecha_alto_menos_bajo_pp',
    'alerta_cobertura_minima_riesgo',
    'alerta_exceso_capacidad',
]].sort_values('ranking_tecnico').round(3))


## Visualizaciones


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
policy_order = df_resumen_equidad.sort_values('ranking_tecnico')['politica'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

coverage_plot = (
    df_equidad_riesgo
    .pivot(index='politica', columns='categoria_riesgo', values='cobertura_media')
    .reindex(index=policy_order, columns=['bajo', 'medio', 'alto'])
)
coverage_plot.plot(kind='bar', ax=axes[0], color=['#72B7B2', '#F2CF5B', '#E45756'])
axes[0].set_title('Cobertura por nivel de riesgo')
axes[0].set_xlabel('Política')
axes[0].set_ylabel('Cobertura media (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Riesgo')

gap_plot = df_brechas_riesgo.set_index('politica').reindex(policy_order)
gap_plot['brecha_absoluta_riesgo_pp'].plot(kind='bar', ax=axes[1], color='#4C78A8')
axes[1].set_title('Brecha de cobertura entre niveles de riesgo')
axes[1].set_xlabel('Política')
axes[1].set_ylabel('Brecha absoluta (puntos porcentuales)')
axes[1].tick_params(axis='x', rotation=45)

figure_path = FIGURES_DIR / 'equidad_cobertura_por_riesgo_v1.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figura guardada:', figure_path)


## Guardar resultados


In [ ]:
STEPS_OUTPUT_PATH = TABLES_DIR / 'trazas_equidad_riesgo_modelos_v1.csv'
VALIDATION_OUTPUT_PATH = TABLES_DIR / 'validacion_pasos_equidad_riesgo_v1.csv'
RISK_SEED_OUTPUT_PATH = TABLES_DIR / 'equidad_por_riesgo_semilla_modelos_v1.csv'
RISK_OUTPUT_PATH = TABLES_DIR / 'equidad_por_riesgo_modelos_v1.csv'
RISK_GAPS_OUTPUT_PATH = TABLES_DIR / 'brechas_por_riesgo_modelos_v1.csv'
HIGH_RISK_OUTPUT_PATH = TABLES_DIR / 'alto_riesgo_por_semilla_modelos_v1.csv'
EQUITY_SUMMARY_PATH = TABLES_DIR / 'resumen_equidad_riesgo_modelos_v1.csv'

df_pasos.to_csv(STEPS_OUTPUT_PATH, index=False, encoding='utf-8-sig')
df_validacion_pasos.to_csv(VALIDATION_OUTPUT_PATH, index=False, encoding='utf-8-sig')
df_riesgo_semilla.to_csv(RISK_SEED_OUTPUT_PATH, index=False, encoding='utf-8-sig')
df_equidad_riesgo.to_csv(RISK_OUTPUT_PATH, index=False, encoding='utf-8-sig')
df_brechas_riesgo.to_csv(RISK_GAPS_OUTPUT_PATH, index=False, encoding='utf-8-sig')
df_alto_riesgo_semilla.to_csv(HIGH_RISK_OUTPUT_PATH, index=False, encoding='utf-8-sig')
df_resumen_equidad.to_csv(EQUITY_SUMMARY_PATH, index=False, encoding='utf-8-sig')

equity_metadata = {
    'version': '3.1',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'audit_scope': 'Equidad operacional por nivel de riesgo en resultados agregados por paso.',
    'not_available_in_source_data': [
        'sexo',
        'edad',
        'grupo_edad',
        'identificador_individual',
    ],
    'fairness_metrics': {
        'brecha_absoluta_riesgo_pp': 'max(cobertura_bajo, cobertura_medio, cobertura_alto) - min(cobertura_bajo, cobertura_medio, cobertura_alto)',
        'brecha_alto_menos_bajo_pp': 'cobertura_alto - cobertura_bajo',
        'alerta_cobertura_minima_exploratoria_pp': 10.0,
    },
    'input_step_files': {source_name: str(path) for source_name, path in STEP_FILES.items()},
    'scope_note': 'Los resultados provienen de un simulador y no demuestran equidad clínica, causalidad ni aptitud para despliegue institucional.',
}

metadata_path = METADATA_DIR / 'protocolo_equidad_riesgo_v3_1.json'
with metadata_path.open('w', encoding='utf-8') as file:
    json.dump(equity_metadata, file, ensure_ascii=False, indent=2)

output_paths = [
    STEPS_OUTPUT_PATH,
    VALIDATION_OUTPUT_PATH,
    RISK_SEED_OUTPUT_PATH,
    RISK_OUTPUT_PATH,
    RISK_GAPS_OUTPUT_PATH,
    HIGH_RISK_OUTPUT_PATH,
    EQUITY_SUMMARY_PATH,
    figure_path,
    metadata_path,
]

for path in output_paths:
    size = path.stat().st_size if path.is_file() else 0
    print(path.name, '| existe:', path.is_file(), '| bytes:', size)

print('Auditoría de equidad por nivel de riesgo completada y guardada.')


## Limitaciones

Los archivos de origen no incluyen sexo, edad, grupo etario ni identificador individual. Por ello no se deben presentar resultados por sexo o edad como si hubieran sido calculados. Una auditoría demográfica futura requerirá trazas individuales con esas variables y un identificador sintético de paciente.
